# Notebook 1 - CNN Parking-Space Classifier (Occupied vs Vacant)

The CNN looks at **one parking space at a time** (a small cropped image) and outputs `P(occupied)`.
The Streamlit app cuts a full parking-lot photo into per-space crops and runs this CNN on each crop.

Steps: load dataset -> clean -> split **without leakage** -> train -> evaluate -> save `cnn_model.keras`.

**Dataset:** use the *segmented* version of **PKLot** (already cropped parking-space images in `Occupied` / `Empty` folders),
or any dataset of per-space crops arranged the same way (e.g. CNRPark+EXT patches sorted into occupied/vacant folders).
Check the dataset page for its **licence and citation requirements** before using it in a report
(PKLot: de Almeida et al., *PKLot - A robust dataset for parking lot classification*, Expert Systems with Applications, 2015).

## Step 1 - Setup

In [ ]:
import os, re, json, hashlib, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ---------------- SETTINGS ----------------
DATA_ROOT = "/content/PKLotSegmented"   # folder that contains the dataset (see Step 2)
IMG_SIZE = 64                            # MUST equal IMG_SIZE in cnn_model.py
MAX_PER_CLASS = 12000                    # cap per class so Colab training stays fast
EPOCHS = 15
BATCH_SIZE = 64

## Step 2 - Get the dataset into Colab

Pick ONE option, then set `DATA_ROOT` above.

* **Option A - Google Drive:** download PKLotSegmented from the official PKLot page, put the archive on your Drive, then run the cell below.
* **Option B - direct link:** paste a direct download link from the dataset page into `DATASET_URL`.

Folder layout expected (PKLotSegmented): `.../<lot>/<weather>/<date>/Occupied/*.jpg` and `.../Empty/*.jpg`.
Any layout works if the image path contains a folder named `Occupied` or `Empty`/`Vacant`/`Free`.

In [ ]:
DATASET_URL = ""            # Option B: paste a direct .tar.gz / .zip link here
DRIVE_ARCHIVE = ""          # Option A: e.g. "/content/drive/MyDrive/PKLotSegmented.tar.gz"

if DRIVE_ARCHIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p {DATA_ROOT} && tar -xf "{DRIVE_ARCHIVE}" -C {DATA_ROOT} --strip-components=0
elif DATASET_URL:
    !wget -q -O /content/dataset_archive "{DATASET_URL}"
    !mkdir -p {DATA_ROOT} && (tar -xf /content/dataset_archive -C {DATA_ROOT} || unzip -q /content/dataset_archive -d {DATA_ROOT})
print("DATA_ROOT exists:", os.path.exists(DATA_ROOT))

## Step 3 - Find all images and their labels

Label comes from the folder name. The **group** is the folder that contains the Occupied/Empty folders
(for PKLot: one lot + weather + day). We split by group in Step 5, so frames of the same day never appear in both train and test.

In [ ]:
OCC_WORDS, VAC_WORDS = {"occupied"}, {"empty", "vacant", "free"}
records = []
for dirpath, _, filenames in os.walk(DATA_ROOT):
    parts = [p.lower() for p in os.path.relpath(dirpath, DATA_ROOT).split(os.sep)]
    label = 1 if OCC_WORDS & set(parts) else 0 if VAC_WORDS & set(parts) else None
    if label is None:
        continue
    class_idx = max(i for i, p in enumerate(parts) if p in OCC_WORDS | VAC_WORDS)
    group = "/".join(os.path.relpath(dirpath, DATA_ROOT).split(os.sep)[:class_idx]) or "root"
    for fn in filenames:
        if fn.lower().endswith((".jpg", ".jpeg", ".png")):
            records.append((os.path.join(dirpath, fn), label, group))
meta = pd.DataFrame(records, columns=["path", "label", "group"])
print(len(meta), "images in", meta["group"].nunique(), "groups")
print(meta["label"].value_counts().rename({0: "vacant", 1: "occupied"}))
assert len(meta) > 0, "No images found - check DATA_ROOT and the folder names."

## Step 4 - Clean the data

1. Random sample (up to `MAX_PER_CLASS` per class) so loading stays fast
2. Remove **corrupted / unreadable / tiny** images (a file that PIL cannot open)
3. Remove **exact duplicates** (same file content)
4. Remove **conflicting labels** - identical image content labelled both occupied and vacant is clearly mislabeled
   (subtle mislabeling cannot be detected automatically)

Images are loaded and resized with **PIL bilinear** - exactly what `cnn_model.py` does - and kept as 0-255 pixels
(the model rescales them itself).

In [ ]:
sample = pd.concat([meta[meta["label"] == lab].sample(min((meta["label"] == lab).sum(), MAX_PER_CLASS), random_state=SEED)
                    for lab in (0, 1)]).reset_index(drop=True)

def load_patch(path):
    with Image.open(path) as im:
        im.load()
        if min(im.size) < 8:
            raise ValueError("too small")
        return np.asarray(im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype="uint8")

arrays, keep, hashes = [], [], []
n_bad = 0
for row in sample.itertuples():
    try:
        arr = load_patch(row.path)
    except Exception:
        n_bad += 1
        continue
    arrays.append(arr); keep.append(row.Index)
    with open(row.path, "rb") as f:
        hashes.append(hashlib.md5(f.read()).hexdigest())
data = sample.loc[keep].reset_index(drop=True)
data["hash"] = hashes
X_all = np.stack(arrays)
print(f"Loaded {len(data)} images; corrupted/unreadable/tiny removed: {n_bad}")

# conflicting labels
conflict = data.groupby("hash")["label"].nunique()
conflict_hashes = set(conflict[conflict > 1].index)
mask = ~data["hash"].isin(conflict_hashes)
print("Removed for conflicting labels:", int((~mask).sum()))
data, X_all = data[mask].reset_index(drop=True), X_all[mask.values]

# exact duplicates
dup = data.duplicated("hash", keep="first")
print("Removed exact duplicates:", int(dup.sum()))
data, X_all = data[~dup].reset_index(drop=True), X_all[~dup.values]
print("Clean dataset:", X_all.shape, "| class counts:", data["label"].value_counts().to_dict())

## Step 5 - Train / validation / test split without leakage

We split **by group** (lot + weather + day), not by image. Consecutive frames of the same day look almost identical,
so a random split would put near-copies in both train and test and give fake, too-high accuracy.
We then assert that no group and no file content appears in two splits.

In [ ]:
groups = data["group"].values
assert data["group"].nunique() >= 6, "Need at least 6 groups (days/lots) to split without leakage."

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, rest_idx = next(gss.split(data, groups=groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_rel, test_rel = next(gss2.split(data.iloc[rest_idx], groups=groups[rest_idx]))
val_idx, test_idx = rest_idx[val_rel], rest_idx[test_rel]

for a, b, name in [(train_idx, val_idx, "train/val"), (train_idx, test_idx, "train/test"), (val_idx, test_idx, "val/test")]:
    assert not set(groups[a]) & set(groups[b]), f"group leakage {name}"
    assert not set(data["hash"].values[a]) & set(data["hash"].values[b]), f"content leakage {name}"
print("No leakage between splits")

X_train, y_train = X_all[train_idx], data["label"].values[train_idx]
X_val, y_val = X_all[val_idx], data["label"].values[val_idx]
X_test, y_test = X_all[test_idx], data["label"].values[test_idx]
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s}: {len(y):6d} images | occupied {y.mean():.1%}")

## Step 6 - Build the CNN

Input: a 64x64 RGB crop. The model itself contains:
* `Rescaling(1/255)` - so the app only passes raw pixels (same preprocessing in training and inference)
* light augmentation (flip, brightness, contrast) - active **only during training**
* 3 convolution blocks -> global pooling -> dense -> sigmoid = **P(occupied)**

In [ ]:
L = tf.keras.layers
model = tf.keras.Sequential([
    L.Input((IMG_SIZE, IMG_SIZE, 3)),
    L.RandomFlip("horizontal"),
    L.RandomBrightness(0.2, value_range=(0, 255)),
    L.RandomContrast(0.2),
    L.Rescaling(1 / 255.0),
    L.Conv2D(32, 3, padding="same", activation="relu"), L.BatchNormalization(), L.MaxPooling2D(),
    L.Conv2D(64, 3, padding="same", activation="relu"), L.BatchNormalization(), L.MaxPooling2D(),
    L.Conv2D(128, 3, padding="same", activation="relu"), L.BatchNormalization(), L.MaxPooling2D(),
    L.GlobalAveragePooling2D(),
    L.Dropout(0.3),
    L.Dense(64, activation="relu"),
    L.Dense(1, activation="sigmoid"),        # P(occupied)   (label 1 = occupied, 0 = vacant)
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## Step 7 - Train (validation set used for early stopping only)

In [ ]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
class_weight = {0: len(y_train) / (2 * neg), 1: len(y_train) / (2 * pos)}      # handles class imbalance
callbacks = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_loss")]
history = model.fit(X_train.astype("float32"), y_train, validation_data=(X_val.astype("float32"), y_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, class_weight=class_weight,
                    callbacks=callbacks, verbose=2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(history.history["accuracy"], label="train"); ax[0].plot(history.history["val_accuracy"], label="validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history.history["loss"], label="train"); ax[1].plot(history.history["val_loss"], label="validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## Step 8 - Evaluate on the test set (images from days the model never saw)

Threshold 0.5, the same as `cnn_model.py`. Test images come from different days of the *same cameras*, so real-world
accuracy on a **different** car park will usually be lower - say so in your presentation.

In [ ]:
probs = model.predict(X_test.astype("float32"), verbose=0).ravel()
pred = (probs >= 0.5).astype(int)
print("Test accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, target_names=["vacant", "occupied"], digits=4))
cm = confusion_matrix(y_test, pred)
ConfusionMatrixDisplay(cm, display_labels=["vacant", "occupied"]).plot(cmap="Blues"); plt.show()

## Step 9 - Save the model and verify the saved file

In [ ]:
os.makedirs("artifacts", exist_ok=True)
model.save("artifacts/cnn_model.keras")

m2 = tf.keras.models.load_model("artifacts/cnn_model.keras", compile=False)
assert np.allclose(m2.predict(X_test[:20].astype("float32"), verbose=0).ravel(), probs[:20], atol=1e-4), "reloaded model differs"

rep = classification_report(y_test, pred, target_names=["vacant", "occupied"], output_dict=True)
metrics = {"img_size": IMG_SIZE, "test_accuracy": float(accuracy_score(y_test, pred)),
           "precision_occupied": rep["occupied"]["precision"], "recall_occupied": rep["occupied"]["recall"],
           "f1_occupied": rep["occupied"]["f1-score"], "n_train": int(len(y_train)), "n_test": int(len(y_test)),
           "dataset_root": DATA_ROOT, "confusion_matrix": cm.tolist()}
json.dump(metrics, open("artifacts/cnn_metrics.json", "w"), indent=2)
print("Saved and verified:", os.listdir("artifacts"))

## Step 10 - Download
Put `cnn_model.keras` into the project's `models/` folder. `cnn_metrics.json` is your evidence for the report (not needed by the app).

In [ ]:
try:
    from google.colab import files
    files.download("artifacts/cnn_model.keras"); files.download("artifacts/cnn_metrics.json")
except ImportError:
    print("Not running in Colab - files are in the 'artifacts' folder.")

### How the app uses this model on a full-lot photo
`cnn_model.py` cuts the uploaded photo into a grid (or the boxes in `data/parking_spaces.json`), resizes each crop to 64x64 with
PIL bilinear (as here), and calls this model. The grid must line up with the real bays - the admin sees the overlay before submitting.